# Análisis previo al agrupamiento K-means

Usamos solo los archivos versionados en `main`:

- `outputs/transformed/X_nhanes.csv` → 48 710 × 42
- `outputs/transformed/y_nhanes.csv` → 48 710 etiquetas

**Regla que seguimos:** la etiqueta no entra al entrenamiento. La guardamos para el final y con ella describimos los grupos. No modificamos ningún archivo de entrada; solo escribimos reportes nuevos.

In [1]:
from pathlib import Path
import time
import tracemalloc

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

raiz = Path.cwd()
if not (raiz / "outputs/transformed/X_nhanes.csv").exists():
    raiz = raiz.parent

X = pd.read_csv(raiz / "outputs/transformed/X_nhanes.csv")
y = pd.read_csv(raiz / "outputs/transformed/y_nhanes.csv")["Disease_Indicator"]

print("X:", X.shape, "| y:", y.shape)
print("NaN:", int(X.isna().sum().sum()), "| infinitos:", int(np.isinf(X.to_numpy(float)).sum()))
print("filas duplicadas:", int(X.duplicated().sum()))
print("columnas constantes:", sum(X[c].nunique() <= 1 for c in X.columns))
print("prevalencia y:", round(float(y.mean()), 4))

X: (48710, 42) | y: (48710,)
NaN: 0 | infinitos: 0
filas duplicadas: 152
columnas constantes: 0
prevalencia y: 0.4111


## 1. Estructura de las variables

Tenemos 10 numéricas ya escaladas (media ≈ 0, desviación ≈ 1) y 32 dummies repartidas en 7 grupos one-hot. Cada grupo suma exactamente 1 por fila.

In [2]:
dummies = X.select_dtypes(include=["int64"]).columns.tolist()
numericas = X.select_dtypes(include=["float64"]).columns.tolist()

grupos = {}
for col in dummies:
    pref = col.rsplit("_", 1)[0]
    grupos.setdefault(pref, []).append(col)

filas = []
for nombre, cols in grupos.items():
    suma = X[cols].sum(axis=1)
    filas.append({"variable": nombre, "n_dummies": len(cols),
                  "prevalencia_min": round(float(X[cols].mean().min()), 4),
                  "prevalencia_max": round(float(X[cols].mean().max()), 4),
                  "suma_min": int(suma.min()), "suma_max": int(suma.max())})

pd.DataFrame(filas).set_index("variable")

,n_dummies,prevalencia_min,prevalencia_max,suma_min,suma_max
variable,,,,,
RIAGENDR,2,0.4950,0.5050,1,1
RIDRETH1,5,0.0924,0.3773,1,1
DMDEDUC2,6,0.0659,0.4168,1,1
DMDMARTL,7,0.0194,0.4164,1,1
RIDSTATR,2,0.0363,0.9637,1,1
HUQ010,5,0.0299,0.3192,1,1
SDDSRVYR,5,0.1922,0.2074,1,1


## 2. Redundancia y señales escondidas

Miramos correlaciones altas entre numéricas y un detalle importante: `DMDEDUC2_0.0` funciona como proxy de menores de edad. Eso explica que aparezca un grupo muy joven.

In [3]:
corr = X[numericas].corr().abs()
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
pares = corr.where(mask).stack().sort_values(ascending=False)
print("Pares |r| >= 0.5:")
print(pares[pares >= 0.5].head(8).round(3).to_string())

menores = X["DMDEDUC2_0.0"] == 1
print("\nDMDEDUC2_0=1:", int(menores.sum()),
      "| edad z media:", round(float(X.loc[menores, "RIDAGEYR"].mean()), 3))
print("DMDEDUC2_0=0:", int((~menores).sum()),
      "| edad z media:", round(float(X.loc[~menores, "RIDAGEYR"].mean()), 3))

Pares |r| >= 0.5:
DR1TKCAL  DR1TTFAT    0.888
          DR1TCARB    0.887
BMXWT     BMXWAIST    0.869
DR1TSUGR  DR1TCARB    0.851
DR1TKCAL  DR1TPROT    0.801
DR1TPROT  DR1TTFAT    0.745
BMXWT     BMXHT       0.736
BMXHT     BMXWAIST    0.677

DMDEDUC2_0=1: 20301 | edad z media: -0.956
DMDEDUC2_0=0: 28409 | edad z media: 0.683


## 3. Conjuntos de variables que comparamos

- **A_todo (42):** todas, incluidas `SDDSRVYR` y `RIDSTATR`.
- **B_sin_tecnicas (35):** quitamos esas dos variables de diseño de encuesta.
- **C_recomendado (30):** lo anterior y además quitamos una dummy de referencia por grupo.
- **D_solo_numericas (10):** control con solo numéricas.

Descartamos `SDDSRVYR` y `RIDSTATR` porque describen el ciclo/entrevista, no a la persona. Quitar una referencia evita contar dos veces la misma categoría en la distancia.

In [4]:
TECNICAS = ["RIDSTATR", "SDDSRVYR"]
referencias = []
for variable in grupos:
    if variable not in TECNICAS:
        referencias.append(sorted(grupos[variable])[-1])

sets = {
    "A_todo": X.columns.tolist(),
    "B_sin_tecnicas": [c for c in X if c.rsplit("_", 1)[0] not in TECNICAS],
    "C_recomendado": [c for c in X if c.rsplit("_", 1)[0] not in TECNICAS and c not in referencias],
    "D_solo_numericas": numericas,
}
print("Referencias que quitamos:", referencias)
print("Tamaños:", {k: len(v) for k, v in sets.items()})

Referencias que quitamos: ['RIAGENDR_2.0', 'RIDRETH1_5.0', 'DMDEDUC2_5.0', 'DMDMARTL_6.0', 'HUQ010_5.0']
Tamaños: {'A_todo': 42, 'B_sin_tecnicas': 35, 'C_recomendado': 30, 'D_solo_numericas': 10}


## 4. Barrido de k

Corremos K-means con `n_init=10`, `random_state=42` y medimos la silueta sobre 2 000 filas. Usamos esa muestra para no armar una matriz de distancias gigante.

In [5]:
filas = []
for nombre, columnas in sets.items():
    Z = X[columnas].to_numpy(dtype=float)
    for k in range(2, 11):
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        etiquetas = km.fit_predict(Z)
        filas.append({"conjunto": nombre, "k": k,
                      "inercia": round(float(km.inertia_), 1),
                      "silueta": round(float(silhouette_score(Z, etiquetas, sample_size=2000, random_state=42)), 4)})

barrido = pd.DataFrame(filas)
pivot = barrido.pivot(index="k", columns="conjunto", values="silueta")
print(pivot.round(4).to_string())
print("\nMejores k:", pivot.idxmax().to_dict())

conjunto  A_todo  B_sin_tecnicas  C_recomendado  D_solo_numericas
k                                                                
2         0.1766          0.1899         0.1979            0.2857
3         0.1930          0.2089         0.2182            0.2673
4         0.1596          0.1742         0.1532            0.2278
5         0.1229          0.1363         0.1451            0.1999
6         0.1200          0.1340         0.1426            0.1975
7         0.1028          0.1155         0.1216            0.1885
8         0.1029          0.1160         0.1227            0.1864
9         0.1022          0.1150         0.1225            0.1849
10        0.1027          0.1160         0.1231            0.1716

Mejores k: {'A_todo': 3, 'B_sin_tecnicas': 3, 'C_recomendado': 3, 'D_solo_numericas': 2}


## 5. Estabilidad y consumo de memoria

Comprobamos que la silueta no cambie al agrandar la muestra y medimos el pico real de memoria Python.

In [6]:
Z = X[sets["C_recomendado"]].to_numpy(dtype=float)
km = KMeans(n_clusters=3, n_init=10, random_state=42)
etiquetas = km.fit_predict(Z)

for n in [2000, 5000, 10000]:
    print("muestra", n, "→ silueta", round(float(silhouette_score(Z, etiquetas, sample_size=n, random_state=42)), 4))

tracemalloc.start()
km = KMeans(n_clusters=3, n_init=10, random_state=42)
km.fit_predict(Z)
silhouette_score(Z, km.labels_, sample_size=2000, random_state=42)
_, pico = tracemalloc.get_traced_memory()
tracemalloc.stop()
print("Pico Python (MB):", round(pico / 1024 / 1024, 1))

muestra 2000 → silueta 0.2182


muestra 5000 → silueta 0.2212


muestra 10000 → silueta 0.2213


Pico Python (MB): 61.9


## 6. Perfil del agrupamiento propuesto (C, k=3)

Usamos la etiqueta solo aquí, para ver si los grupos tienen perfiles distintos.

In [7]:
perfil = X[sets["C_recomendado"]].copy()
perfil["cluster"] = etiquetas
perfil["y"] = y.values
perfil.groupby("cluster").agg(
    n=("y", "size"),
    prevalencia=("y", "mean"),
    edad_z=("RIDAGEYR", "mean"),
    cintura_z=("BMXWAIST", "mean"),
    proxy_menores=("DMDEDUC2_0.0", "mean"),
).round(4)

,n,prevalencia,edad_z,cintura_z,proxy_menores
cluster,,,,,
0,9130,0.4807,0.0861,0.3138,0.2789
1,23304,0.6371,0.6759,0.5262,0.0689
2,16276,0.0484,-1.0161,-0.9294,0.9922


## 7. Recomendación

- Usamos el conjunto **C_recomendado** (30 columnas).
- Fijamos **k=3**, `random_state=42`, `n_init=10`.
- Reportamos la silueta baja (≈0,22) con honestidad: son perfiles, no grupos perfectamente separados.
- No usamos la etiqueta para entrenar; solo para el perfil.
- No necesitamos GPU: el pico medido fueron unas decenas de MB.

## 8. En palabras simples

Hicimos tres cosas:

1. **Limpiamos la lista de variables.** Quitamos dos columnas que solo describen la encuesta, no a las personas. También quitamos una columna repetida de cada grupo para no contar dos veces lo mismo.
2. **Probamos cuántos grupos conviene.** Probamos de 2 a 9 grupos. Con 3 grupos sale el mejor resultado estable.
3. **Miramos qué significa cada grupo.** Sin usar la etiqueta, aparecieron tres perfiles: personas jóvenes, adultos con mayor prevalencia y un grupo intermedio. Al comparar con la etiqueta vimos que la prevalencia cambia mucho entre grupos, así que los perfiles sirven.

Lo que hacemos ahora es **correr K-means con esas 30 columnas y k=3**, guardar las etiquetas de grupo y escribir la conclusión. No necesitamos más RAM ni tarjeta gráfica.